In [ ]:
!pip install -U langchain-chroma langchain-huggingface langchain-community langchain-text-splitters
!pip install -U langchain-openai
!pip install -U langchain-groq

In [ ]:
import os
import glob
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.messages import SystemMessage, HumanMessage
from google.colab import userdata

In [ ]:
# key=userdata.get('OPENROUTER_API_KEY')
# key=userdata.get('GOOGLE_API_KEY')
key=userdata.get('GROQ_API_KEY')

# url = "https://generativelanguage.googleapis.com/v1beta/openai/"
url = "https://api.groq.com/openai/v1"


# MODEL = "gemini-3.6-flash"
MODEL = "openai/gpt-oss-120b"

In [ ]:
# Load in everything in the knowledgebase using LangChain's loaders

folders = glob.glob("/content/drive/MyDrive/knowledge-base/*")

documents = []
for folder in folders:
    doc_type = os.path.basename(folder)
    loader = DirectoryLoader(folder, glob="**/*.md", loader_cls=TextLoader, loader_kwargs={'encoding': 'utf-8'})
    folder_docs = loader.load()
    for doc in folder_docs:
        doc.metadata["doc_type"] = doc_type
        documents.append(doc)

print(f"Loaded {len(documents)} documents")

In [ ]:
# Divide into chunks using the RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(documents)

print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

In [ ]:
# Pick an embedding model

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
# embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

db_name = "vector_db"

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

In [ ]:
retriever = vectorstore.as_retriever()
retriever.invoke("Who is Avery?")

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(temperature=0, model_name='openai/gpt-oss-120b', api_key=key)
llm.invoke("Who is Avery?")

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [ ]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer_question("Who is Averi Lancaster?", [])

In [ ]:
import gradio as gr
gr.ChatInterface(answer_question).launch()